# Imports

In [11]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

import requests
import json
from google.oauth2 import service_account
from google.auth.transport.requests import Request
from IPython.display import Audio
import base64

# Create Songs Code

In [12]:
creds = service_account.Credentials.from_service_account_file(
    "keys/service.json",
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
creds.refresh(Request())
token = creds.token

with open('keys/keys.json', 'r') as f:
    keys = json.load(f)

In [13]:
# Constants we need
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}
# this is a secret ;)
url = "https://aiplatform.googleapis.com/v1beta1/projects/wpp-cto-smartprod-sandbox/locations/global/interactions"

In [14]:
def create_song(prompt):
    """
    Using parameters create a song using prompt.

    Example:
    prompt = "Om nom nom fireflies eat each other rah rah that's how we get bigger that is how we grow"
    """
    params = { 
        "model": "lyria-3-clip-preview", 
        "input": 
            [
                { 
                  "type": "text", 
                  "text": f"{prompt}"
                }
            ]
        }
    
    response = requests.post(url, headers=headers, json=params)
    data = response.json()
    bytedata = data["outputs"][2]["data"]

    return base64.b64decode(bytedata)

def display_song(bytes):
    """
    make it so we can see the song and listen XD rawr
    """
    return Audio(bytes)

def save_song_file(bytes, fname="audio/downloaded_audio.mp3"):
    """
    saves it to disk :p
    """
    with open(fname, "wb") as file:
        file.write(bytes)
    print(f"Audio file saved successfully as '{fname}'!")
    

In [17]:
def play_announcement(prompt):
    """create and play a song"""
    mp3 = create_song(prompt)
    display(Audio(mp3, autoplay=True))


## Example Usage of Song Creation

In [16]:
# Create Song
song_bytes = create_song("Om nom nom fireflies eat each other rah rah that's how we get bigger that is how we grow")

# save to disk
# save_song_file(song_bytes, "om_nom_nom.mp3")
display_song(song_bytes)

# Game Code

In [2]:
# Load Game Data
df = pd.read_csv("../data/firefly_data_game_dataset_v3.csv")

In [3]:
# Create Player and GUI

# Create a simple player profile starting as a Class A firefly
player_stats = {'mass': 10.0, 'speed': 5.0}

# 2. Build the Widgets (The GUI)
enemy_dropdown = widgets.Dropdown(
    options=['Class A (Mass 10)', 'Class B (Mass 25)', 'Class C (Mass 50)', 'Class D (Boss - Mass 100)'],
    description='Target:',
    disabled=False,
)

hunt_button = widgets.Button(
    description='Hunt Target!',
    button_style='success',
    tooltip='Attempt to eat the target firefly'
)

output_screen = widgets.Output()


In [46]:
# The Game Logic Function

def on_hunt_click(b):
    with output_screen:
        clear_output()
        
        # Determine target mass from the dropdown selection
        target_selection = enemy_dropdown.value
        if "Class A" in target_selection: target_mass = 10
        elif "Class B" in target_selection: target_mass = 25
        elif "Class C" in target_selection: target_mass = 50
        else: target_mass = 100
            
        match = df[df['starting_mass'] == target_mass]
        
        if match.empty:
            print(f"Error: Firefly with {target_mass} is not found in the dataframe.")
            return
            
        enemy = match.iloc[0]
        
        print(f"--- HUNT INITIATED ---")
        print(f"You (Mass: {player_stats['mass']}, Speed: {player_stats['speed']}) are hunting a {target_selection}.")
        print(f"Enemy Speed is {enemy['movement_speed']}.")
        
        # Speed dictates ability to cut off the enemy
        if player_stats['speed'] >= enemy['movement_speed']:
            print(f"\nSUCCESS! You are fast enough to cut off the enemy's head.")
            # Player grows by consuming the food_value
            player_stats['mass'] += enemy['food_value']
            
            # As player grows, their speed decreases (balancing mechanic)
            if player_stats['mass'] >= 50:
                player_stats['speed'] = 3.0
            elif player_stats['mass'] >= 25:
                player_stats['speed'] = 4.0

            # if the boss was defeated play music
            if target_selection[:7] == "Class D":
                play_announcement("(house music) You ate the biggest firefly you are a winner now you are the biggest and you will be eaten.")
                
            print(f"You ate the firefly! Your mass is now {player_stats['mass']}.")
        else:
            print(f"\nFAILURE! The enemy's speed ({enemy['movement_speed']}) is higher than yours.")
            print("They escaped your trap.")

# Play Game

In [47]:
hunt_button.on_click(on_hunt_click)
display(widgets.VBox([enemy_dropdown, hunt_button, output_screen]))